In [ ]:
# ============================================================================
# ANÁLISE DE ERRO — GeoFM v2 etapa5
# ============================================================================
# Objetivo: identificar as condições locais em que o modelo falha.
#
# Perguntas:
#   1. Os erros têm padrão espacial? (por Class8590, por região N/S)
#   2. Os erros têm padrão temporal? (por T — ano de entrada em pastagem)
#   3. Os erros têm padrão de vizinhança? (estado do patch antes de T)
#
# Inputs:
#   - spatial_split_test.csv         (1.505 pixels, labels reais)
#   - multihead_spatial_frozen_*.pth  (modelo congelado etapa5)
#
# Outputs:
#   - predictions_test.csv           (predições por pixel + metadados)
#   - análises visuais inline
# ============================================================================

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Caminhos ────────────────────────────────────────────────────────────────
BASE_DIR   = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
SPLIT_DIR  = BASE_DIR / "spatial_split"
FREEZE_DIR = BASE_DIR / "multihead_spatial_frozen"
LULC_DIR   = r"D:\Projetos\Cerrado\LULC"

TEST_CSV   = SPLIT_DIR  / "spatial_split_test.csv"
MODEL_PTH  = FREEZE_DIR / "multihead_spatial_frozen_20260318_060216.pth"
OUT_CSV    = FREEZE_DIR / "predictions_test.csv"

print(f"✅ Configuração OK!")
print(f"   Device:     {DEVICE}")
print(f"   Test CSV:   {TEST_CSV.name}")
print(f"   Modelo:     {MODEL_PTH.name}")
print(f"   Output CSV: {OUT_CSV.name}")

In [ ]:
# ============================================================================
# ARQUITETURA — cópia exata do etapa5
# ============================================================================

class MultiHeadSpatialModel(nn.Module):

    def __init__(self, input_dim=287, hidden_dim=256, encoder_dim=128,
                 head_hidden_dim=64, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, encoder_dim), nn.ReLU(), nn.Dropout(dropout)
        )
        self.head_disperso = nn.Sequential(
            nn.Linear(encoder_dim, head_hidden_dim), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1), nn.Sigmoid()
        )
        self.head_cluster = nn.Sequential(
            nn.Linear(encoder_dim, head_hidden_dim), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1), nn.Sigmoid()
        )
        gate_input_dim = encoder_dim + 3
        self.gate = nn.Sequential(
            nn.Linear(gate_input_dim, 32), nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(32, 2), nn.Softmax(dim=1)
        )

    def forward(self, x, spatial_features):
        encoded       = self.encoder(x)
        pred_disperso = self.head_disperso(encoded)
        pred_cluster  = self.head_cluster(encoded)
        gate_input    = torch.cat([encoded, spatial_features], dim=1)
        gate_weights  = self.gate(gate_input)
        w_d = gate_weights[:, 0:1]
        w_c = gate_weights[:, 1:2]
        prediction = w_d * pred_disperso + w_c * pred_cluster
        return prediction, gate_weights


# Carregar modelo congelado
checkpoint = torch.load(MODEL_PTH, map_location='cpu')
model = MultiHeadSpatialModel(**checkpoint['config'])
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()

print(f"✅ Modelo carregado!")
print(f"   Parâmetros: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Treinado até epoch: {checkpoint['training']['best_epoch']}")
print(f"   Val accuracy (congelado): "
      f"{checkpoint['metrics']['validation']['accuracy']:.4f}")
print(f"   Test accuracy (congelado): "
      f"{checkpoint['metrics']['test']['accuracy']:.4f}")

In [ ]:
# ============================================================================
# DATASET — idêntico ao etapa5
# ============================================================================

import rasterio
from rasterio.windows import Window
from torch.utils.data import Dataset, DataLoader

DATA_DIR      = LULC_DIR
PATTERN       = "brazil_coverage_{ano}_Cerrado.tif"
YEARS         = list(range(1985, 2025))
NODATA_IN     = 255
NODATA_OUT    = 0
PATCH_N       = 7
MAX_SERIE_LEN = len(YEARS) - 1
PATCH_YEARS   = 5


def _path(year):
    return os.path.join(DATA_DIR, PATTERN.format(ano=year))

def _ler_pixel(year, row, col):
    with rasterio.open(_path(year)) as ds:
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)

def _ler_patch(year, row, col, n=PATCH_N):
    half = n // 2
    with rasterio.open(_path(year)) as ds:
        H, W = ds.height, ds.width
        col0 = min(max(0, col - half), W - n)
        row0 = min(max(0, row - half), H - n)
        arr  = ds.read(1, window=Window(col0, row0, n, n), out_dtype="uint8")
    return np.where(arr == NODATA_IN, NODATA_OUT, arr).astype(np.uint8)


class TestDataset(Dataset):
    """Dataset de inferência — retorna também row, col, T para análise."""

    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
        self.df = self.df[self.df['label'].notna()].reset_index(drop=True)
        self.df['label'] = self.df['label'].astype(int)
        print(f"  Pixels: {len(self.df):,}")
        print(f"  Labels: {self.df['label'].value_counts().to_dict()}")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r     = self.df.iloc[idx]
        row   = int(r['row'])
        col   = int(r['col'])
        T     = int(r['T'])
        label = int(r['label'])

        anos_serie = [y for y in YEARS if y < T]
        if len(anos_serie) > 0:
            serie_raw = np.array([_ler_pixel(y, row, col)
                                  for y in anos_serie], dtype=np.float32)
            serie_raw = np.where(serie_raw == NODATA_IN,
                                 NODATA_OUT, serie_raw).astype(np.float32)
        else:
            serie_raw = np.array([], dtype=np.float32)

        serie_len = len(serie_raw)
        serie     = np.zeros(MAX_SERIE_LEN, dtype=np.float32)
        if serie_len > 0:
            serie[MAX_SERIE_LEN - serie_len:] = serie_raw

        anos_patch = [y for y in YEARS if y < T][-PATCH_YEARS:]
        patch = np.zeros((PATCH_YEARS, PATCH_N, PATCH_N), dtype=np.float32)
        for i, y in enumerate(anos_patch):
            patch[PATCH_YEARS - len(anos_patch) + i] = _ler_patch(y, row, col)

        if serie_len > 0:
            has_21    = float(np.sum(serie_raw == 21)) / serie_len
            anos_past = sum(1 for v in reversed(serie_raw) if v == 15)
            cl_tm1    = float(serie_raw[-1])
        else:
            has_21 = anos_past = cl_tm1 = 0.0

        # Sempre neutro (sem pattern_code no CSV final)
        spatial = np.array([0.5, 0.5, 1.5], dtype=np.float32)

        features = np.concatenate([serie, patch.flatten(),
                                   np.array([has_21, float(anos_past),
                                             cl_tm1], dtype=np.float32)])
        assert features.shape == (287,)

        return (
            torch.tensor(features, dtype=torch.float32),
            torch.tensor(spatial,  dtype=torch.float32),
            torch.tensor([label],  dtype=torch.float32),
            row, col, T
        )


print("Test dataset:")
test_ds = TestDataset(TEST_CSV)
print(f"\n✅ Dataset pronto")

In [ ]:
# ============================================================================
# INFERÊNCIA — gerar predições por pixel
# ============================================================================

# DataLoader com batch_size=1 para preservar row/col/T por amostra
# (DataLoader não lida bem com scalars misturados com tensors em batch)
# Alternativa mais limpa: iterar diretamente

print("Rodando inferência no test set...")
print("⚠️  Lê rasters pixel a pixel — ~10-20 min")
print()

rows_out, cols_out, Ts_out        = [], [], []
labels_out, probs_out             = [], []
w_disperso_out, w_cluster_out     = [], []
pred_class_out                    = []

model.eval()
with torch.no_grad():
    for idx in tqdm(range(len(test_ds)), desc="Inferência"):
        features, spatial, label, row, col, T = test_ds[idx]

        features = features.unsqueeze(0).to(DEVICE)
        spatial  = spatial.unsqueeze(0).to(DEVICE)

        prob, gate_w = model(features, spatial)

        p_val = float(prob.squeeze().cpu())
        w_d   = float(gate_w[0, 0].cpu())
        w_c   = float(gate_w[0, 1].cpu())
        lbl   = int(label.item())
        pred  = 1 if p_val >= 0.5 else 0

        rows_out.append(row)
        cols_out.append(col)
        Ts_out.append(T)
        labels_out.append(lbl)
        probs_out.append(round(p_val, 6))
        pred_class_out.append(pred)
        w_disperso_out.append(round(w_d, 6))
        w_cluster_out.append(round(w_c, 6))

# Montar DataFrame de predições
df_pred = pd.DataFrame({
    'row':          rows_out,
    'col':          cols_out,
    'T':            Ts_out,
    'label':        labels_out,
    'prob':         probs_out,
    'pred':         pred_class_out,
    'correct':      [int(l == p) for l, p in zip(labels_out, pred_class_out)],
    'w_disperso':   w_disperso_out,
    'w_cluster':    w_cluster_out,
})

# Adicionar colunas diagnósticas
df_pred['error_type'] = 'TN'   # true negative
df_pred.loc[(df_pred['label']==1) & (df_pred['pred']==1), 'error_type'] = 'TP'
df_pred.loc[(df_pred['label']==1) & (df_pred['pred']==0), 'error_type'] = 'FN'
df_pred.loc[(df_pred['label']==0) & (df_pred['pred']==1), 'error_type'] = 'FP'

# Adicionar Class8590 do CSV (já disponível)
df_test_meta = pd.read_csv(TEST_CSV)
df_test_meta = df_test_meta[df_test_meta['label'].notna()].reset_index(drop=True)
df_pred['Class8590'] = df_test_meta['Class8590'].values

# Salvar
df_pred.to_csv(OUT_CSV, index=False)

print(f"\n✅ Predições salvas: {OUT_CSV.name}")
print(f"   Shape: {df_pred.shape}")
print(f"\nMatriz de confusão:")
print(df_pred['error_type'].value_counts())

acc = df_pred['correct'].mean()
print(f"\nAccuracy: {acc:.4f}  (esperado: ~0.603)")

In [ ]:
# ============================================================================
# ANÁLISE 1 — PADRÃO ESPACIAL DOS ERROS
# ============================================================================

print("=" * 60)
print("ANÁLISE 1 — PADRÃO ESPACIAL (Class8590)")
print("=" * 60)

# Accuracy por padrão
acc_by_pattern = df_pred.groupby('Class8590').agg(
    n_pixels   = ('correct', 'count'),
    accuracy   = ('correct', 'mean'),
    n_TP = ('error_type', lambda x: (x=='TP').sum()),
    n_FP = ('error_type', lambda x: (x=='FP').sum()),
    n_FN = ('error_type', lambda x: (x=='FN').sum()),
    n_TN = ('error_type', lambda x: (x=='TN').sum()),
    pct_label1 = ('label', 'mean')
).reset_index().sort_values('accuracy')

print(f"\n{'Pattern':<8} {'n':>6} {'Acc':>7} {'%rapid':>7} "
      f"{'TP':>5} {'FP':>5} {'FN':>5} {'TN':>5}")
print("-" * 56)
for _, r in acc_by_pattern.iterrows():
    print(f"{r['Class8590']:<8} {r['n_pixels']:>6} "
          f"{r['accuracy']:>7.3f} {r['pct_label1']:>7.3f} "
          f"{r['n_TP']:>5} {r['n_FP']:>5} "
          f"{r['n_FN']:>5} {r['n_TN']:>5}")

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy por padrão
patterns = acc_by_pattern['Class8590']
accs     = acc_by_pattern['accuracy']
colors   = ['#e74c3c' if a < 0.55 else '#f39c12' if a < 0.65
            else '#27ae60' for a in accs]
axes[0].barh(patterns, accs, color=colors)
axes[0].axvline(x=df_pred['correct'].mean(), color='navy',
                linestyle='--', label=f'Média geral ({acc:.3f})')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Accuracy por padrão espacial (Class8590)')
axes[0].legend()
axes[0].set_xlim(0, 1)

# Distribuição de erros por padrão (stacked)
err_by_pat = df_pred.groupby(['Class8590', 'error_type']).size().unstack(fill_value=0)
err_by_pat = err_by_pat.reindex(columns=['TP','TN','FP','FN'], fill_value=0)
err_by_pat_pct = err_by_pat.div(err_by_pat.sum(axis=1), axis=0) * 100
err_by_pat_pct.plot(kind='barh', stacked=True, ax=axes[1],
                    color=['#27ae60','#2ecc71','#e74c3c','#c0392b'])
axes[1].set_xlabel('% amostras')
axes[1].set_title('Composição de predições por padrão')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig(str(FREEZE_DIR / 'error_analysis_spatial.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Gráfico salvo: error_analysis_spatial.png")

In [ ]:
# ============================================================================
# ANÁLISE 2 — PADRÃO TEMPORAL DOS ERROS
# ============================================================================

print("=" * 60)
print("ANÁLISE 2 — PADRÃO TEMPORAL (T = ano de entrada em pastagem)")
print("=" * 60)

# Accuracy por T (agrupado em períodos de 5 anos)
df_pred['T_period'] = (df_pred['T'] // 5) * 5
acc_by_T = df_pred.groupby('T_period').agg(
    n_pixels = ('correct', 'count'),
    accuracy = ('correct', 'mean'),
    pct_FN   = ('error_type', lambda x: (x=='FN').sum() / len(x)),
    pct_FP   = ('error_type', lambda x: (x=='FP').sum() / len(x)),
    pct_label1 = ('label', 'mean')
).reset_index()

print(f"\n{'Período':<10} {'n':>6} {'Acc':>7} {'%rapid':>7} "
      f"{'%FN':>7} {'%FP':>7}")
print("-" * 50)
for _, r in acc_by_T.iterrows():
    print(f"{int(r['T_period'])}-{int(r['T_period'])+4:<6} "
          f"{r['n_pixels']:>6} {r['accuracy']:>7.3f} "
          f"{r['pct_label1']:>7.3f} {r['pct_FN']:>7.3f} {r['pct_FP']:>7.3f}")

# Gráfico
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Accuracy por período
axes[0].bar(acc_by_T['T_period'], acc_by_T['accuracy'],
            color='steelblue', width=4)
axes[0].axhline(y=acc, color='red', linestyle='--',
                label=f'Média ({acc:.3f})')
axes[0].set_xlabel('Ano de entrada em pastagem (T)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy por período de T')
axes[0].legend()

# FN por período
axes[1].bar(acc_by_T['T_period'], acc_by_T['pct_FN'],
            color='#e74c3c', width=4, label='FN (conversão perdida)')
axes[1].set_xlabel('Ano de entrada em pastagem (T)')
axes[1].set_ylabel('% Falsos Negativos')
axes[1].set_title('Taxa de FN por período de T')
axes[1].legend()

# Distribuição de amostras por período
axes[2].bar(acc_by_T['T_period'], acc_by_T['n_pixels'],
            color='gray', width=4, alpha=0.7)
axes[2].set_xlabel('Ano de entrada em pastagem (T)')
axes[2].set_ylabel('N pixels')
axes[2].set_title('Distribuição de amostras por período')

plt.tight_layout()
plt.savefig(str(FREEZE_DIR / 'error_analysis_temporal.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Gráfico salvo: error_analysis_temporal.png")

In [ ]:
# ============================================================================
# ANÁLISE 3 — DISTRIBUIÇÃO GEOGRÁFICA DOS ERROS
# ============================================================================
# Usa row/col como coordenadas de raster (proxy geográfico)
# row alto = sul do Cerrado, col alto = leste

print("=" * 60)
print("ANÁLISE 3 — DISTRIBUIÇÃO GEOGRÁFICA (row/col)")
print("=" * 60)

# Dividir em quadrantes N/S e L/O
row_median = df_pred['row'].median()
col_median = df_pred['col'].median()

df_pred['region_NS'] = df_pred['row'].apply(
    lambda r: 'Norte' if r < row_median else 'Sul')
df_pred['region_LO'] = df_pred['col'].apply(
    lambda c: 'Oeste' if c < col_median else 'Leste')

print(f"\nRow mediana: {row_median:.0f} (Norte=acima, Sul=abaixo)")
print(f"Col mediana: {col_median:.0f} (Oeste=abaixo, Leste=acima)")

acc_by_region = df_pred.groupby(['region_NS', 'region_LO']).agg(
    n       = ('correct', 'count'),
    acc     = ('correct', 'mean'),
    pct_FN  = ('error_type', lambda x: (x=='FN').sum()/len(x)),
    pct_FP  = ('error_type', lambda x: (x=='FP').sum()/len(x)),
).reset_index()

print(f"\n{'Região':<15} {'n':>6} {'Acc':>7} {'%FN':>7} {'%FP':>7}")
print("-" * 45)
for _, r in acc_by_region.iterrows():
    print(f"{r['region_NS']+'-'+r['region_LO']:<15} "
          f"{r['n']:>6} {r['acc']:>7.3f} "
          f"{r['pct_FN']:>7.3f} {r['pct_FP']:>7.3f}")

# Scatter plot geográfico
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_map = {'TP': '#27ae60', 'TN': '#2ecc71',
              'FP': '#e74c3c', 'FN': '#c0392b'}

for etype, color in colors_map.items():
    sub = df_pred[df_pred['error_type'] == etype]
    axes[0].scatter(sub['col'], -sub['row'], c=color, s=6,
                    alpha=0.5, label=etype)
axes[0].set_xlabel('Col (proxy Leste →)')
axes[0].set_ylabel('Row (proxy Norte ↑)')
axes[0].set_title('Distribuição espacial dos erros')
axes[0].legend(markerscale=3)

# Só erros
for etype, color in [('FP', '#e74c3c'), ('FN', '#c0392b')]:
    sub = df_pred[df_pred['error_type'] == etype]
    axes[1].scatter(sub['col'], -sub['row'], c=color, s=8,
                    alpha=0.6, label=etype)
axes[1].set_xlabel('Col (proxy Leste →)')
axes[1].set_ylabel('Row (proxy Norte ↑)')
axes[1].set_title('Apenas erros: FP e FN')
axes[1].legend(markerscale=3)

plt.tight_layout()
plt.savefig(str(FREEZE_DIR / 'error_analysis_geographic.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Gráfico salvo: error_analysis_geographic.png")

In [ ]:
# ============================================================================
# ANÁLISE 4 — CONFIANÇA DO MODELO
# ============================================================================
# O modelo é conservador ou agressivo?
# Onde ele erra, estava confiante ou incerto?

print("=" * 60)
print("ANÁLISE 4 — CONFIANÇA DO MODELO (distribuição de probabilidades)")
print("=" * 60)

# Estatísticas de probabilidade por tipo de erro
prob_stats = df_pred.groupby('error_type')['prob'].describe()
print(f"\nProbabilidades por tipo de predição:")
print(prob_stats[['count','mean','std','min','25%','50%','75%','max']]
      .round(3).to_string())

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de probabilidades por tipo
for etype, color, alpha in [
    ('TP', '#27ae60', 0.7), ('TN', '#3498db', 0.7),
    ('FP', '#e74c3c', 0.7), ('FN', '#8e44ad', 0.7)
]:
    sub = df_pred[df_pred['error_type'] == etype]['prob']
    axes[0].hist(sub, bins=20, alpha=alpha, color=color,
                 label=f'{etype} (n={len(sub)})')
axes[0].axvline(x=0.5, color='black', linestyle='--', label='Threshold 0.5')
axes[0].set_xlabel('Probabilidade predita')
axes[0].set_ylabel('N pixels')
axes[0].set_title('Distribuição de probabilidades por tipo de erro')
axes[0].legend()

# Calibração: prob predita vs taxa real de acerto
df_pred['prob_bin'] = pd.cut(df_pred['prob'], bins=10)
calib = df_pred.groupby('prob_bin', observed=True).agg(
    frac_pos = ('label', 'mean'),
    mean_prob = ('prob', 'mean'),
    n = ('label', 'count')
).reset_index()

axes[1].scatter(calib['mean_prob'], calib['frac_pos'],
                s=calib['n']*2, alpha=0.7, color='steelblue')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.5, label='Calibração perfeita')
axes[1].set_xlabel('Probabilidade média predita')
axes[1].set_ylabel('Fração real de positivos')
axes[1].set_title('Calibração do modelo')
axes[1].legend()
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(str(FREEZE_DIR / 'error_analysis_confidence.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Gráfico salvo: error_analysis_confidence.png")

In [ ]:
# ============================================================================
# RESUMO DIAGNÓSTICO
# ============================================================================

print("=" * 60)
print("RESUMO DIAGNÓSTICO")
print("=" * 60)

total    = len(df_pred)
n_TP     = (df_pred['error_type']=='TP').sum()
n_TN     = (df_pred['error_type']=='TN').sum()
n_FP     = (df_pred['error_type']=='FP').sum()
n_FN     = (df_pred['error_type']=='FN').sum()
n_errors = n_FP + n_FN

print(f"\nTotal amostras: {total}")
print(f"  TP: {n_TP:4d} ({100*n_TP/total:.1f}%)")
print(f"  TN: {n_TN:4d} ({100*n_TN/total:.1f}%)")
print(f"  FP: {n_FP:4d} ({100*n_FP/total:.1f}%) — prediz conversão, não converte")
print(f"  FN: {n_FN:4d} ({100*n_FN/total:.1f}%) — não prediz, mas converte")
print(f"\nTotal erros: {n_errors} ({100*n_errors/total:.1f}%)")

# Padrão mais problemático
worst = acc_by_pattern.iloc[0]
best  = acc_by_pattern.iloc[-1]
print(f"\nPadrão com pior accuracy:  "
      f"{worst['Class8590']} ({worst['accuracy']:.3f}, n={worst['n_pixels']})")
print(f"Padrão com melhor accuracy: "
      f"{best['Class8590']} ({best['accuracy']:.3f}, n={best['n_pixels']})")

# Período mais problemático
worst_T = acc_by_T.sort_values('accuracy').iloc[0]
print(f"\nPeríodo com pior accuracy: "
      f"{int(worst_T['T_period'])}-{int(worst_T['T_period'])+4} "
      f"(acc={worst_T['accuracy']:.3f}, n={worst_T['n_pixels']})")

# FN vs FP dominância
print(f"\nTipo de erro dominante: "
      f"{'FN' if n_FN > n_FP else 'FP'} "
      f"({max(n_FN,n_FP)} vs {min(n_FN,n_FP)})")
print(f"  → O modelo é "
      f"{'CONSERVADOR (subestima conversão)' if n_FN > n_FP else 'AGRESSIVO (superestima conversão)'}")

print(f"\n{'='*60}")
print(f"Predições salvas em: {OUT_CSV}")
print(f"Gráficos salvos em:  {FREEZE_DIR}")
print(f"{'='*60}")